# ForgeAudio (`audio-forge`) — Remote A100/L4 GPU Worker

Runs GPU stem separation (`BS-RoFormer 1297` / `BS-RoFormer-SW` / `vocals_mel_band_roformer.ckpt` / `MDX23C DrumSep` / `Karaoke` / `De-Bleed` / `De-Aspiration`), denoising, neural restoration, vocal rescue, beat tracking, transcription, and perceptual quality evaluation on Google Colab and serves `POST /api/v1/execute` for `forge` and `forge-mcp` (`FORGE_COLAB_WORKER_URL`).

In [ ]:
!pip install -q fastapi uvicorn[standard] "audio-separator[gpu]" soundfile librosa numpy pyngrok huggingface_hub basic-pitch torch torchaudio
!git clone --depth 1 https://github.com/The-40-Thieves/audio-forge.git /content/audio-forge 2>/dev/null || git -C /content/audio-forge pull --ff-only

In [ ]:
import base64
import hmac
import os
import sys
import tempfile
from pathlib import Path
from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel
import torch

WORKER_DIR = Path(os.environ.get("FORGE_WORKER_DIR", "/content/audio-forge/python-worker"))
if str(WORKER_DIR) not in sys.path:
    sys.path.insert(0, str(WORKER_DIR))
import forge_gpu_worker as worker

app = FastAPI(title="ForgeAudio Colab A100 Worker")
MAX_UPLOAD_BYTES = 256 * 1024 * 1024
API_TOKEN = os.environ.get("FORGE_COLAB_API_TOKEN", "").strip()
if not API_TOKEN:
    raise RuntimeError("FORGE_COLAB_API_TOKEN must be set before starting the worker")

def require_bearer(authorization: str | None = Header(default=None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Missing Bearer token")
    token = authorization[7:].strip()
    if not hmac.compare_digest(token, API_TOKEN):
        raise HTTPException(status_code=403, detail="Invalid Bearer token")

class WorkerRequest(BaseModel):
    task: str
    payload: dict

def _decode_input_to(work: Path, b64: str | None, filename: str, stem_prefix: str = "input") -> Path:
    if not isinstance(b64, str) or len(b64) > (MAX_UPLOAD_BYTES * 4) // 3 + 4:
        raise HTTPException(status_code=413, detail="Encoded audio payload missing or too large")
    try:
        raw = base64.b64decode(b64, validate=True)
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid base64 audio payload: {exc}") from exc
    if len(raw) > MAX_UPLOAD_BYTES:
        raise HTTPException(status_code=413, detail="Decoded audio exceeds 256 MiB limit")
    ext = "".join(c for c in Path(str(filename)).suffix.lower() if c.isalnum())
    dest = (work / f"{stem_prefix}.{ext or 'wav'}").resolve()
    if not dest.is_relative_to(work):
        raise HTTPException(status_code=400, detail="Invalid filename")
    dest.write_bytes(raw)
    return dest

def _encode_artifact(work: Path, path_str: str) -> str:
    resolved = Path(path_str).resolve()
    if not resolved.is_relative_to(work) or not resolved.is_file():
        raise HTTPException(status_code=500, detail="Worker wrote output outside temp directory")
    return base64.b64encode(resolved.read_bytes()).decode("ascii")

@app.get("/health")
def health():
    return {
        "ok": True,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else 0,
        "tasks": ["separate_stems", "denoise", "transcribe", "restore", "vocal_rescue", "beat_track", "evaluate_quality"]
    }

@app.post("/api/v1/execute", dependencies=[Depends(require_bearer)])
def execute(req: WorkerRequest):
    p = req.payload
    with tempfile.TemporaryDirectory(prefix="forge_") as tmp:
        work = Path(tmp).resolve()
        out_dir = (work / "out").resolve()
        out_dir.mkdir(parents=True, exist_ok=True)
        src = _decode_input_to(work, p.get("audio_b64"), str(p.get("filename", "input.wav")))

        if req.task == "separate_stems":
            raw = worker.run_separate(
                src, out_dir,
                dereverb=bool(p.get("dereverb", True)),
                stems=int(p.get("stems", 2)),
                quality=str(p.get("quality", "studio")),
                drum_sep=bool(p.get("drum_sep", False)),
                split_lead_backing=bool(p.get("split_lead_backing", False)),
                split_male_female=bool(p.get("split_male_female", False)),
                split_guitar=bool(p.get("split_guitar", False)),
                split_orchestral=bool(p.get("split_orchestral", False)),
                split_crowd=bool(p.get("split_crowd", False)),
                restore=bool(p.get("restore", False)),
                debleed=bool(p.get("debleed", False)),
                de_aspiration=bool(p.get("de_aspiration", False)),
                dereverb_echo=bool(p.get("dereverb_echo", False)),
            )
            warnings = raw.pop("warnings", []) or []
            stems = {k: _encode_artifact(out_dir, str(v)) for k, v in raw.items() if isinstance(v, str)}
            if not stems:
                raise HTTPException(status_code=500, detail="separator produced no stems")
            return {"ok": True, "task": req.task, "data": {"stems": stems, "warnings": warnings}}

        if req.task == "denoise":
            res = worker.run_denoise(src, out_dir, attenuation_limit_db=p.get("attenuation_limit_db"), backend=str(p.get("backend", "auto")))
            return {"ok": True, "task": req.task, "data": {"audio_b64": _encode_artifact(out_dir, res["denoised"]), "backend": res["backend"]}}

        if req.task == "transcribe":
            res = worker.run_transcribe(src, out_dir)
            return {"ok": True, "task": req.task, "data": {"midi_b64": _encode_artifact(out_dir, res["midi"])}}

        if req.task == "restore":
            res = worker.run_restore(src, out_dir, backend=str(p.get("backend", "auto")), prompt=p.get("prompt"), chunk_secs=float(p.get("chunk_secs", 10.0)))
            return {"ok": True, "task": req.task, "data": {"audio_b64": _encode_artifact(out_dir, str(res["restored"])), "backend": res["backend"], "warnings": res["warnings"]}}

        if req.task == "vocal_rescue":
            ref = _decode_input_to(work, p.get("reference_voice_b64"), str(p.get("reference_filename", "ref.wav")), "ref") if p.get("reference_voice_b64") else None
            res = worker.run_vocal_rescue(src, out_dir, backend=str(p.get("backend", "auto")), reference_voice=ref, dereverb=bool(p.get("dereverb", True)), de_aspiration=bool(p.get("de_aspiration", True)))
            return {"ok": True, "task": req.task, "data": {"audio_b64": _encode_artifact(out_dir, str(res["rescued"])), "backend": res["backend"], "stages": res["stages"], "warnings": res["warnings"]}}

        if req.task == "beat_track":
            res = worker.run_beat_track(src, backend=str(p.get("backend", "auto")))
            return {"ok": True, "task": req.task, "data": res}

        if req.task == "evaluate_quality":
            ref = _decode_input_to(work, p.get("reference_b64"), str(p.get("reference_filename", "ref.wav")), "eval_ref") if p.get("reference_b64") else None
            res = worker.run_evaluate(src, reference_path=ref, vocal=bool(p.get("vocal", False)), backend=str(p.get("backend", "auto")))
            return {"ok": True, "task": req.task, "data": res}

        raise HTTPException(status_code=400, detail=f"Unknown task: {req.task}")